# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane: Growth / Recovery / Momentum Prediction** (freestyle direction, warehouse daily facts).

> Skills loaded for this notebook: `writing-data-contracts/SKILL.md`, `querying-big-datasets/SKILL.md`, `hunting-leakage-and-validating/SKILL.md`, `flyrank/flyrank-data/SKILL.md` (per `skills/README.md`).

## 1. Unit of analysis + time window

**One row = one (client, content item) pair's performance inside March 2026**, built from `fact_content_daily_performance`, partition `month=2026-03` (a mid-panel month — never the `_sample` table, which is the sealed final month, June 2026).

**Table(s) used:** `fact_content_daily_performance` (grain: `report_date × client_hash_id × content_hash_id`), joined for context only to `dim_clients` (grain: one row per client) to read `gsc_data_start` / `ga4_data_start`.

**Time window, split in half inside the one partition (this is the decision point):**
- **Feature window (“before”):** `report_date` 2026-03-01 → 2026-03-15 (15 days, trailing, already happened)
- **Label window (“after”):** `report_date` 2026-03-16 → 2026-03-31 (15 days, strictly future relative to the decision point)

I'm using a half-month split instead of the production 90-day/30-day windows so the whole exercise stays inside one partition (no multi-month joins, no rate-limit risk) — flagged again as a named limitation in section 4.

In [ ]:
# ---- Setup: DuckDB over the remote warehouse (never download it) ----
%pip -q install duckdb

import duckdb

# Colab Secrets panel (key icon, left sidebar) -> add a secret named HF_TOKEN.
# NEVER paste a token directly into a cell -- this repo is public.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HF_TOKEN (plain Read token, gated-repositories permission ticked): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month -- iterate here, never on the _sample (final, sealed) month
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"
CLIENTS = f"read_parquet('{BASE}/dim_clients.parquet')"

# --- Schema discovery FIRST. I have confirmed report_date, client_hash_id, content_hash_id,
# gsc_avg_position, ga4_data_available from the repo docs. I have NOT independently verified the
# exact impressions/clicks/sessions column names below -- read the printed schema and fix COLS
# in the next cell if any name doesn't match what you see. ---
schema = con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 1").df()
print(schema.to_string())


In [ ]:
# --- ASSUMPTION TO CONFIRM: adjust these four names against the DESCRIBE output above ---
COLS = {
    "impressions": "gsc_impressions",   # <-- confirm/edit against the schema printed above
    "clicks": "gsc_clicks",             # <-- confirm/edit against the schema printed above
    "position": "gsc_avg_position",     # confirmed in docs/data-dictionary.md
    "ga4_flag": "ga4_data_available",   # confirmed in docs/data-dictionary.md -- filter with IS TRUE, never = TRUE
}

missing = [c for c in [COLS['impressions'], COLS['clicks'], COLS['position'], COLS['ga4_flag']]
           if c not in schema['column_name'].values]
assert not missing, f"Fix COLS -- these names were not found in the schema: {missing}"
print("COLS verified against schema:", COLS)


## 2. Fields: feature / label / context / excluded

**Target / proxy:** `is_declining_next_half` — 1 if a (client, content) pair's summed impressions in the label window (Mar 16–31) drop more than 20% versus the feature window (Mar 1–15), else 0. This mirrors the starter's `trend_direction == "down"` rule, but on a real **future** window instead of a same-window bucket — an observed outcome, not a current label.

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `impressions_first_half`, `clicks_first_half`, `ctr_first_half`, `avg_position_first_half`, `active_days_first_half` | all aggregated only from Mar 1–15 — knowable at the Mar 15 decision point |
| **Label / proxy** | `impressions_second_half` (used only to compute the label) → `is_declining_next_half` | Mar 16–31, strictly after the decision point |
| **Context** | `client_hash_id`, `content_hash_id` | joins, grouping, client-holdout splits — never features |
| **Excluded** | `impressions_second_half` itself as a feature | it *is* the label's raw material — including it is the leakage trap I deliberately demonstrate and then remove in section 3 |

Also excluded on principle: any `ga4_*` metric for rows where `ga4_data_available IS NOT TRUE` (zero-filled, not "no engagement" — see section 3's availability check).

In [ ]:
# Print the bucket table as a quick sanity dataframe (no query needed here -- this section is
# the plan; section 3 is where every claim above gets checked against real rows).
import pandas as pd
buckets = pd.DataFrame([
    ("impressions_first_half", "feature"),
    ("clicks_first_half", "feature"),
    ("ctr_first_half", "feature"),
    ("avg_position_first_half", "feature"),
    ("active_days_first_half", "feature"),
    ("impressions_second_half", "label material (excluded as a feature)"),
    ("is_declining_next_half", "label"),
    ("client_hash_id", "context"),
    ("content_hash_id", "context"),
], columns=["field", "bucket"])
buckets


## 3. Verify it with queries (grain, counts, availability) + five features + the leakage trap

### 3a. Grain check
One row of `fact_content_daily_performance` really is one `report_date × client_hash_id × content_hash_id`. Zero rows back below means the grain holds.

In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate (date, client, content) combinations found: {len(grain_check)}")
grain_check


### 3b. Slice row count and date span
Row count, distinct clients/content, and the date span for `month=2026-03`. `COUNT(*)` and `MIN/MAX(date)` are near-free on Parquet metadata.

In [ ]:
slice_stats = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        COUNT(DISTINCT content_hash_id)   AS n_content,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date
    FROM {FACT}
""").df()
slice_stats


### 3c. Availability check (filter with `IS TRUE`)
`ga4_data_available` is three-valued (TRUE / FALSE / NULL) — filtering with `= TRUE` or `NOT ...` silently mishandles the NULL rows. Using `IS TRUE` shows exactly how many rows actually survive.

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {COLS['ga4_flag']} IS TRUE  THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN {COLS['ga4_flag']} IS FALSE THEN 1 ELSE 0 END) AS ga4_unavailable_rows,
        SUM(CASE WHEN {COLS['ga4_flag']} IS NULL  THEN 1 ELSE 0 END) AS ga4_null_rows,
        ROUND(100.0 * SUM(CASE WHEN {COLS['ga4_flag']} IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {FACT}
""").df()
availability


### 3d. Build the five-feature frame

All five features are aggregated **only** from the Mar 1–15 feature window — nothing here reads a single row from Mar 16–31.

| Feature | Available when? |
|---|---|
| `impressions_first_half` | knowable at the Mar 15 decision point — sums days that already happened |
| `clicks_first_half` | same — trailing sum over the feature window only |
| `ctr_first_half` | computed purely from the two features above, both already-observed |
| `avg_position_first_half` | average of a daily field that is recorded as each day occurs, over already-elapsed days only |
| `active_days_first_half` | counts already-elapsed days with impressions > 0 — no future days touched |

In [ ]:
feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM({COLS['impressions']}) AS impressions_first_half,
            SUM({COLS['clicks']})      AS clicks_first_half,
            AVG(CASE WHEN {COLS['impressions']} > 0 THEN {COLS['position']} END) AS avg_position_first_half,
            COUNT(DISTINCT CASE WHEN {COLS['impressions']} > 0 THEN report_date END) AS active_days_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM({COLS['impressions']}) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id, f.content_hash_id,
        f.impressions_first_half,
        f.clicks_first_half,
        ROUND(100.0 * f.clicks_first_half / NULLIF(f.impressions_first_half, 0), 2) AS ctr_first_half,
        f.avg_position_first_half,
        f.active_days_first_half,
        COALESCE(s.impressions_second_half, 0) AS impressions_second_half,
        CASE
            WHEN f.impressions_first_half > 0
                 AND (COALESCE(s.impressions_second_half, 0) - f.impressions_first_half)
                     / f.impressions_first_half < -0.20
            THEN 1 ELSE 0
        END AS is_declining_next_half
    FROM first_half f
    LEFT JOIN second_half s USING (client_hash_id, content_hash_id)
    WHERE f.impressions_first_half > 0
""").df()

print(f"Feature frame: {len(feature_frame):,} (client, content) pairs")
print(f"Base rate of is_declining_next_half: {feature_frame['is_declining_next_half'].mean():.1%}")
feature_frame.head()


### 3e. The trap — add a label-derived column on purpose, watch the score jump, then remove it

`impressions_second_half` is the exact raw material the label is computed from. Training WITH it vs WITHOUT it is the notebook-02 leakage lesson, done on real warehouse data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                    "avg_position_first_half", "active_days_first_half"]
leaky_features = honest_features + ["impressions_second_half"]

df = feature_frame.dropna(subset=honest_features).copy()
y = df["is_declining_next_half"]

# grouped-ish split by client so no content leaks between train/test (a lightweight version of
# the grouped split -- ML-09 does this properly with GroupKFold)
train_clients, test_clients = train_test_split(df["client_hash_id"].unique(), test_size=0.3, random_state=42)
train_mask = df["client_hash_id"].isin(train_clients)
test_mask = df["client_hash_id"].isin(test_clients)

def quick_score(feature_cols):
    X_train, X_test = df.loc[train_mask, feature_cols], df.loc[test_mask, feature_cols]
    y_train, y_test = y[train_mask], y[test_mask]
    model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, preds)

auc_honest = quick_score(honest_features)
auc_leaky = quick_score(leaky_features)

print(f"Honest 5-feature AUC (no label-derived column): {auc_honest:.3f}")
print(f"Leaky 6th-feature AUC (impressions_second_half included): {auc_leaky:.3f}")
print("\n-> The leaky version jumps toward 1.0 because impressions_second_half basically IS the")
print("   label's raw material. Deleting it and keeping the honest number below.")

# --- delete the leak, keep the honest number ---
final_auc = auc_honest
print(f"\nFinal, honest score kept for this notebook: AUC = {final_auc:.3f}")


## 4. Data limits

**Named limitation:** the 15-day/15-day feature/label split is a scaled-down stand-in for the production-style 90-day/30-day windows — it fits inside one `month=2026-03` partition on purpose, but low-volume (client, content) pairs get noisy first-half aggregates over only 15 days, and the 20%-drop threshold is more sensitive to day-to-day noise than a 90-day baseline would be. On top of that: the panel is unbalanced (per-client history depth differs — `dim_clients.gsc_data_start`/`ga4_data_start` should be checked before trusting this window for any one client), and rows with `ga4_data_available` NULL (neither zero-filled nor flagged FALSE) are silently dropped from any GA4-based feature unless filtered with `IS TRUE` as done in 3c.

In [ ]:
# Quick evidence for the panel-imbalance limitation: how much per-client GSC history existed
# BEFORE this slice's month even starts.
history_check = con.sql(f"""
    SELECT
        MIN(gsc_data_start) AS earliest_client_start,
        MAX(gsc_data_start) AS latest_client_start,
        COUNT(*) AS n_clients
    FROM {CLIENTS}
""").df()
history_check


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.